# Step 12: Train Best Model (Part 4)
**Model:** SP Large (33.6M parameters)
**Target:** 10 epochs with a tuned stable learning rate of `3e-4`

**Instructions:**
1. Run this notebook on an A100 instance.
2. Make sure `train.npy` and `val.npy` are on your Google Drive at `MyDrive/ML_project/data/splits/`.
3. After completion, download `best_model.pt` and metrics.json.

In [ ]:
import os, sys, json, time, math, shutil
from pathlib import Path
from dataclasses import dataclass
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F

from google.colab import drive
drive.mount('/content/drive')

BASE = Path("/content/ML_project"); BASE.mkdir(exist_ok=True)
DATA = BASE/"data"/"splits"; DATA.mkdir(parents=True, exist_ok=True)
CKPT = BASE/"checkpoints"/"best_model"; CKPT.mkdir(parents=True, exist_ok=True)

DRIVE_DATA = Path("/content/drive/MyDrive/ML_project/data/splits")
for f in ["train.npy", "val.npy"]:
    src, dst = DRIVE_DATA/f, DATA/f
    if src.exists() and not dst.exists(): shutil.copy2(src, dst); print(f"Copied {f}")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB")

In [ ]:
@dataclass
class GPTConfig:
    block_size: int = 2048
    vocab_size: int = 4096
    n_layer: int = 10     # Large model
    n_head: int = 8       # Large model
    n_embd: int = 512     # Large model
    d_ff: int = 2048      # Large model
    dropout: float = 0.0  # no dropout during pre-training scale
    bias: bool = False

class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=None,
                                           dropout_p=self.dropout if self.training else 0,
                                           is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, config.d_ff, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(config.d_ff, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight # Weight tying
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None: torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def get_num_params(self, non_embedding=True):
        n = sum(p.numel() for p in self.parameters())
        if non_embedding: n -= self.transformer.wpe.weight.numel()
        return n

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        pos = torch.arange(0, t, dtype=torch.long, device=device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss = None
        return logits, loss

In [ ]:
def get_batch(data, block_size, batch_size, device):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+block_size].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def evaluate(model, data, block_size, batch_size, eval_batches, device, ctx):
    model.eval()
    losses = []
    for _ in range(eval_batches):
        x, y = get_batch(data, block_size, batch_size, device)
        with ctx:
            _, loss = model(x, y)
            losses.append(loss.item())
    model.train()
    return float(np.mean(losses))

def get_lr(step, warmup_steps, max_steps, max_lr, min_lr):
    if step < warmup_steps: return max_lr * (step + 1) / warmup_steps
    if step >= max_steps: return min_lr
    decay_ratio = (step - warmup_steps) / (max_steps - warmup_steps)
    return min_lr + 0.5 * (1.0 + math.cos(math.pi * decay_ratio)) * (max_lr - min_lr)

In [ ]:
device = "cuda"
ctx = torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16)

# Training Hyperparameters
epochs = 10
learning_rate = 3e-4 # Tuned down from 3e-3 for stability
micro_batch_size = 32 # Large batch for A100
gradient_accum_steps = 8 # 32 * 8 = 256 batch size (524K tokens/step)
warmup_steps = 1000 # Longer warmup for 10 epochs

model = GPT(GPTConfig()).to(device)
print(f"Created Large Model: {model.get_num_params():,} parameters")

# Optimizer
param_dict = {pn: p for pn, p in model.named_parameters() if p.requires_grad}
decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
optim_groups = [
    {'params': decay_params, 'weight_decay': 0.1},
    {'params': nodecay_params, 'weight_decay': 0.0}
]
optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=(0.9, 0.95), fused=True)
scaler = torch.amp.GradScaler(enabled=False) # Bfloat16 doesn't need scaling

In [ ]:
train_data = np.load(DATA / "train.npy", mmap_mode='r')
val_data = np.load(DATA / "val.npy", mmap_mode='r')

block_size = 2048
tokens_per_step = micro_batch_size * gradient_accum_steps * block_size
steps_per_epoch = len(train_data) // tokens_per_step
total_steps = steps_per_epoch * epochs

print(f"Training for {epochs} epochs ({total_steps} steps)")
print(f"Tokens per step: {tokens_per_step:,}")

best_val_loss = float('inf')
train_losses, val_losses = [], []
t0 = time.time()
total_tokens = 0

for step in range(total_steps):
    lr = get_lr(step, warmup_steps, total_steps, learning_rate, learning_rate * 0.1)
    for param_group in optimizer.param_groups: param_group['lr'] = lr
    
    optimizer.zero_grad(set_to_none=True)
    accum_loss = 0.0
    
    for _ in range(gradient_accum_steps):
        x, y = get_batch(train_data, block_size, micro_batch_size, device)
        with ctx:
            _, loss = model(x, y)
            loss = loss / gradient_accum_steps
        scaler.scale(loss).backward()
        accum_loss += loss.item()
        total_tokens += x.numel()
        
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    
    if (step + 1) % 10 == 0:
        train_losses.append((step, accum_loss))
        
    if (step + 1) % 100 == 0 or step == total_steps - 1:
        val_loss = evaluate(model, val_data, block_size, micro_batch_size, 20, device, ctx)
        val_losses.append((step, val_loss))
        
        # Save checkpoint if it's the best
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            ckpt_path = CKPT / "model.pt"
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'step': step,
                'val_loss': val_loss,
            }, ckpt_path)
            # Copy directly to drive as well
            drive_ckpt = Path("/content/drive/MyDrive/ML_project/checkpoints/best_model")
            drive_ckpt.mkdir(parents=True, exist_ok=True)
            shutil.copy2(ckpt_path, drive_ckpt / "model.pt")
            saved_msg = " [Saved Checkpoint]"
        else:
            saved_msg = ""
            
        print(f"Epoch {(step+1)/steps_per_epoch:.2f} | Step {step+1}/{total_steps} | LR {lr:.6f} | Train {accum_loss:.4f} | Val {val_loss:.4f}{saved_msg}")

wall_time = time.time() - t0
print(f"\nTraining Complete in {wall_time/3600:.2f} hours!")
print(f"Best Validation Loss: {best_val_loss:.4f}")

# Save final metrics
metrics = {
    "model": "large_sp",
    "epochs": epochs,
    "total_steps": total_steps,
    "best_val_loss": best_val_loss,
    "train_losses": train_losses,
    "val_losses": val_losses,
    "wall_time_seconds": wall_time
}
with open(CKPT / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
shutil.copy2(CKPT / "metrics.json", Path("/content/drive/MyDrive/ML_project/checkpoints/best_model/metrics.json"))

print("All results saved to Google Drive.")